In [223]:
  %pip install pytrends praw google-api-python-client python-dotenv requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\gablima1\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


CONFIGURAÇÃO DAS CHAVES — PROJETO HOT TOPICS


In [ ]:
# YouTube (já temos!)
YOUTUBE_API_KEY = "Cole_seu_codigo_aqui"

# Reddit (preencher depois)
REDDIT_CLIENT_ID = ""
REDDIT_CLIENT_SECRET = ""
REDDIT_USER_AGENT = "HotTopics/1.0 by Simple_Appearance495"

# Meta (preencher depois — aguardando 24h)
META_ACCESS_TOKEN = "COLE_AQUI_SUA_CHAVE"

# X/Twitter (preencher depois — requer plano pago)
TWITTER_BEARER_TOKEN = ""

# TikTok (preencher depois — requer plano pago)
TIKTOK_API_KEY = ""

print("✅ Configuração carregada!")
print(f"YouTube: {'✅ Configurado' if YOUTUBE_API_KEY else '❌ Pendente'}")
print(f"Reddit:  {'✅ Configurado' if REDDIT_CLIENT_ID else '⏳ Pendente'}")
print(f"Meta:    {'✅ Configurado' if META_ACCESS_TOKEN else '⏳ Pendente'}")
print(f"Twitter: {'✅ Configurado' if TWITTER_BEARER_TOKEN else '⏳ Pendente'}")
print(f"TikTok:  {'✅ Configurado' if TIKTOK_API_KEY else '⏳ Pendente'}")

✅ Configuração carregada!
YouTube: ✅ Configurado
Reddit:  ⏳ Pendente
Meta:    ✅ Configurado
Twitter: ⏳ Pendente
TikTok:  ⏳ Pendente


TESTE YOUTUBE

In [225]:
from googleapiclient.discovery import build

def coletar_youtube_trending(api_key, regiao="BR", max_resultados=10):
    youtube = build("youtube", "v3", developerKey=api_key)

    resposta = youtube.videos().list(
        part="snippet,statistics",
        chart="mostPopular",
        regionCode=regiao,
        maxResults=max_resultados
    ).execute()

    topicos = []
    for video in resposta["items"]:
        titulo = video["snippet"]["title"]
        canal = video["snippet"]["channelTitle"]
        views = video["statistics"].get("viewCount", 0)
        likes = video["statistics"].get("likeCount", 0)

        topicos.append({
            "titulo": titulo,
            "canal": canal,
            "views": int(views),
            "likes": int(likes)
        })

    return topicos

# Testar!
resultados = coletar_youtube_trending(YOUTUBE_API_KEY)

print(f"🎬 TOP {len(resultados)} VÍDEOS EM ALTA NO BRASIL\n")
for i, video in enumerate(resultados, 1):
    print(f"{i}. {video['titulo']}")
    print(f"   Canal: {video['canal']}")
    print(f"   Views: {int(video['views']):,} | Likes: {int(video['likes']):,}")
    print()

🎬 TOP 10 VÍDEOS EM ALTA NO BRASIL

1. Separa e Volta - MC Vine7, MC Tuto, MC Ryan SP, MC Joãozinho VT e MC Poze do Rodo (DJ Gu)
   Canal: MC Vine7
   Views: 88,112 | Likes: 2,493

2. Faces of Grief - Release Date Trailer
   Canal: The Walten Files
   Views: 106,277 | Likes: 18,000

3. NUNCA VENHA NESSA HAMBURGUERIA! - HAPPY'S HUMBLE BURGER CULT COM OS AMIGOS
   Canal: alanzoka
   Views: 363,471 | Likes: 26,006

4. ♪ Spider-Noir (Homem-Aranha) | Whisky & Teias | AniRap, @TakaB e @AnnyTHN
   Canal: AniRap
   Views: 160,849 | Likes: 27,406

5. House of the Dragon: S3 - Ep.06 | NEW TRAILER | HBO
   Canal: AD_edits
   Views: 68,246 | Likes: 587

6. PROCURA-SE MESSI - ESPANHA CAMPEÃ!!! FREE FIRE E ALGO MAIS VEM
   Canal: Apelapato999
   Views: 190,478 | Likes: 12,215

7. BTS (방탄소년단) ‘NORMAL’ Official MV
   Canal: HYBE LABELS
   Views: 5,270,931 | Likes: 930,703

8. Akira | Relançamento em 4K | Trailer Oficial
   Canal: ingresso.com
   Views: 87,606 | Likes: 6,458

9. Fazendo perguntas BIZAR

TESTE GOOGLE TRENDS

In [226]:
import requests
import xml.etree.ElementTree as ET

def coletar_google_trends(max_resultados=10):
    url = "https://trends.google.com/trending/rss?geo=BR"

    headers = {"User-Agent": "Mozilla/5.0"}
    resposta = requests.get(url, headers=headers)

    root = ET.fromstring(resposta.content)

    topicos = []
    for item in root.findall(".//item")[:max_resultados]:
        titulo = item.find("title").text
        topicos.append(titulo)

    print(f"🔍 TOP {len(topicos)} TENDÊNCIAS NO GOOGLE BRASIL\n")
    for i, topico in enumerate(topicos, 1):
        print(f"{i}. {topico}")

    return topicos

resultados_trends = coletar_google_trends()

🔍 TOP 10 TENDÊNCIAS NO GOOGLE BRASIL

1. deborah secco
2. joão sanches
3. memphis depay
4. voto em transito
5. andy burnham
6. clima
7. what
8. wpp
9. star wars
10. paraná pesquisas


TESTE REDDIT

In [227]:
import praw

def coletar_reddit(client_id="", client_secret="", user_agent="", max_resultados=10):
    if not client_id or not client_secret:
        print("⏳ Reddit: credenciais não configuradas — pulando coleta")
        return []

    reddit = praw.Reddit(
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent
    )

    subreddits_br = ["brasil", "investimentos", "futebol", "tecnologia", "noticias"]

    topicos = []
    for sub in subreddits_br:
        for post in reddit.subreddit(sub).hot(limit=max_resultados//len(subreddits_br)):
            topicos.append({
                "titulo": post.title,
                "subreddit": sub,
                "upvotes": post.score,
                "comentarios": post.num_comments,
                "url": post.url
            })

    print(f"✅ Reddit: {len(topicos)} tópicos coletados")
    return topicos

resultados_reddit = coletar_reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT
)

⏳ Reddit: credenciais não configuradas — pulando coleta


TESTE META

In [228]:
import requests

def coletar_meta(access_token="", max_resultados=10):
    if not access_token:
        print("⏳ Meta: credenciais não configuradas — pulando coleta")
        return []

    topicos = []

    # ── Instagram ────────────────────────────────────────────────
    try:
        url_ig = f"https://graph.facebook.com/v22.0/me/media"
        params = {
            "fields": "id,caption,like_count,comments_count,timestamp",
            "limit": max_resultados,
            "access_token": access_token
        }
        resposta = requests.get(url_ig, params=params)
        dados = resposta.json()

        for post in dados.get("data", []):
            topicos.append({
                "plataforma": "instagram",
                "titulo": post.get("caption", "")[:100],
                "likes": post.get("like_count", 0),
                "comentarios": post.get("comments_count", 0),
                "data": post.get("timestamp", "")
            })
    except Exception as e:
        print(f"  Instagram erro: {e}")

    # ── Facebook Pages ────────────────────────────────────────────
    try:
        url_fb = f"https://graph.facebook.com/v22.0/me/posts"
        params = {
            "fields": "id,message,likes.summary(true),comments.summary(true),created_time",
            "limit": max_resultados,
            "access_token": access_token
        }
        resposta = requests.get(url_fb, params=params)
        dados = resposta.json()

        for post in dados.get("data", []):
            topicos.append({
                "plataforma": "facebook",
                "titulo": post.get("message", "")[:100],
                "likes": post.get("likes", {}).get("summary", {}).get("total_count", 0),
                "comentarios": post.get("comments", {}).get("summary", {}).get("total_count", 0),
                "data": post.get("created_time", "")
            })
    except Exception as e:
        print(f"  Facebook erro: {e}")

    print(f"✅ Meta: {len(topicos)} posts coletados")
    return topicos

resultados_meta = coletar_meta(access_token=META_ACCESS_TOKEN)

✅ Meta: 0 posts coletados


Instalar X/Twitter

In [229]:
import requests

def coletar_twitter(bearer_token="", max_resultados=10):
    if not bearer_token:
        print("⏳ Twitter/X: credenciais não configuradas — pulando coleta")
        return []

    headers = {"Authorization": f"Bearer {bearer_token}"}

    topicos = []

    # ── Trending Topics Brasil ────────────────────────────────────
    try:
        # Woeid 455189 = Brasil
        url = "https://api.twitter.com/1.1/trends/place.json"
        params = {"id": 455189}
        resposta = requests.get(url, headers=headers, params=params)
        dados = resposta.json()

        trends = dados[0]["trends"][:max_resultados]

        for trend in trends:
            topicos.append({
                "plataforma": "twitter",
                "titulo": trend["name"],
                "volume": trend.get("tweet_volume", 0) or 0,
                "url": trend.get("url", "")
            })

        print(f"✅ Twitter/X: {len(topicos)} trending topics coletados")

    except Exception as e:
        print(f"  Twitter/X erro: {e}")

    return topicos

resultados_twitter = coletar_twitter(bearer_token=TWITTER_BEARER_TOKEN)

⏳ Twitter/X: credenciais não configuradas — pulando coleta


Instalar TikTok

In [230]:
import requests

def coletar_tiktok(api_key="", max_resultados=10):
    if not api_key:
        print("⏳ TikTok: credenciais não configuradas — pulando coleta")
        return []

    topicos = []

    try:
        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }

        # ── Trending Hashtags ─────────────────────────────────────
        url = "https://open.tiktokapis.com/v2/research/hashtag/query/"
        params = {
            "fields": "hashtag_name,view_count,video_count"
        }
        resposta = requests.get(url, headers=headers, params=params)
        dados = resposta.json()

        for hashtag in dados.get("data", {}).get("hashtags", [])[:max_resultados]:
            topicos.append({
                "plataforma": "tiktok",
                "titulo": f"#{hashtag.get('hashtag_name', '')}",
                "views": hashtag.get("view_count", 0),
                "videos": hashtag.get("video_count", 0)
            })

        print(f"✅ TikTok: {len(topicos)} hashtags coletadas")

    except Exception as e:
        print(f"  TikTok erro: {e}")

    return topicos

resultados_tiktok = coletar_tiktok(api_key=TIKTOK_API_KEY)

⏳ TikTok: credenciais não configuradas — pulando coleta


Consolidar Plataformas

In [231]:
def consolidar_todas_plataformas():
    print("🚀 HOT TOPICS — Iniciando coleta em todas as plataformas...\n")

    todos_topicos = []

    # ── Google Trends ─────────────────────────────────────────────
    print("📡 Coletando Google Trends...")
    trends = coletar_google_trends()
    for t in trends:
        todos_topicos.append({
            "plataforma": "google_trends",
            "titulo": t,
            "metrica_principal": 0,
            "metrica_secundaria": 0
        })

    # ── YouTube ───────────────────────────────────────────────────
    print("\n📡 Coletando YouTube...")
    youtube = coletar_youtube_trending(YOUTUBE_API_KEY)
    for t in youtube:
        todos_topicos.append({
            "plataforma": "youtube",
            "titulo": t["titulo"],
            "metrica_principal": t["views"],
            "metrica_secundaria": t["likes"]
        })

    # ── Reddit ────────────────────────────────────────────────────
    print("\n📡 Coletando Reddit...")
    reddit = coletar_reddit(REDDIT_CLIENT_ID, REDDIT_CLIENT_SECRET, REDDIT_USER_AGENT)
    for t in reddit:
        todos_topicos.append({
            "plataforma": "reddit",
            "titulo": t["titulo"],
            "metrica_principal": t["upvotes"],
            "metrica_secundaria": t["comentarios"]
        })

    # ── Meta ──────────────────────────────────────────────────────
    print("\n📡 Coletando Meta...")
    meta = coletar_meta(META_ACCESS_TOKEN)
    for t in meta:
        todos_topicos.append({
            "plataforma": t["plataforma"],
            "titulo": t["titulo"],
            "metrica_principal": t["likes"],
            "metrica_secundaria": t["comentarios"]
        })

    # ── Twitter/X ─────────────────────────────────────────────────
    print("\n📡 Coletando Twitter/X...")
    twitter = coletar_twitter(TWITTER_BEARER_TOKEN)
    for t in twitter:
        todos_topicos.append({
            "plataforma": "twitter",
            "titulo": t["titulo"],
            "metrica_principal": t["volume"],
            "metrica_secundaria": 0
        })

    # ── TikTok ────────────────────────────────────────────────────
    print("\n📡 Coletando TikTok...")
    tiktok = coletar_tiktok(TIKTOK_API_KEY)
    for t in tiktok:
        todos_topicos.append({
            "plataforma": "tiktok",
            "titulo": t["titulo"],
            "metrica_principal": t["views"],
            "metrica_secundaria": t["videos"]
        })

    print(f"\n{'='*50}")
    print(f"✅ COLETA CONCLUÍDA!")
    print(f"Total de tópicos coletados: {len(todos_topicos)}")
    print(f"Plataformas ativas: {len(set(t['plataforma'] for t in todos_topicos))}/6")
    print(f"{'='*50}")

    return todos_topicos

# Executar!
todos_topicos = consolidar_todas_plataformas()

🚀 HOT TOPICS — Iniciando coleta em todas as plataformas...

📡 Coletando Google Trends...
🔍 TOP 10 TENDÊNCIAS NO GOOGLE BRASIL

1. deborah secco
2. joão sanches
3. memphis depay
4. voto em transito
5. andy burnham
6. clima
7. what
8. wpp
9. star wars
10. paraná pesquisas

📡 Coletando YouTube...

📡 Coletando Reddit...
⏳ Reddit: credenciais não configuradas — pulando coleta

📡 Coletando Meta...
✅ Meta: 0 posts coletados

📡 Coletando Twitter/X...
⏳ Twitter/X: credenciais não configuradas — pulando coleta

📡 Coletando TikTok...
⏳ TikTok: credenciais não configuradas — pulando coleta

✅ COLETA CONCLUÍDA!
Total de tópicos coletados: 20
Plataformas ativas: 2/6


Classificação P,M E G

In [232]:
def classificar_pmg(topicos):
    print("\n🧠 Classificando tópicos em P / M / G...\n")

    classificados = []

    for topico in topicos:
        plataforma = topico["plataforma"]
        metrica = topico["metrica_principal"]

        # ── Regras de classificação por plataforma ────────────────
        if plataforma == "google_trends":
            # Google Trends: posição no ranking (todos são relevantes)
            classificacao = "P"  # Emergente por padrão no Trends

        elif plataforma == "youtube":
            if metrica >= 5_000_000:
                classificacao = "G"  # Mainstream
            elif metrica >= 500_000:
                classificacao = "M"  # Crescendo
            else:
                classificacao = "P"  # Emergente

        elif plataforma == "reddit":
            if metrica >= 10_000:
                classificacao = "G"
            elif metrica >= 1_000:
                classificacao = "M"
            else:
                classificacao = "P"

        elif plataforma in ["instagram", "facebook"]:
            if metrica >= 100_000:
                classificacao = "G"
            elif metrica >= 10_000:
                classificacao = "M"
            else:
                classificacao = "P"

        elif plataforma == "twitter":
            if metrica >= 100_000:
                classificacao = "G"
            elif metrica >= 10_000:
                classificacao = "M"
            else:
                classificacao = "P"

        elif plataforma == "tiktok":
            if metrica >= 10_000_000:
                classificacao = "G"
            elif metrica >= 1_000_000:
                classificacao = "M"
            else:
                classificacao = "P"
        else:
            classificacao = "P"

        classificados.append({
            **topico,
            "classificacao": classificacao
        })

    # ── Resumo ────────────────────────────────────────────────────
    p = [t for t in classificados if t["classificacao"] == "P"]
    m = [t for t in classificados if t["classificacao"] == "M"]
    g = [t for t in classificados if t["classificacao"] == "G"]

    print(f"🌱 P — Emergentes:  {len(p)} tópicos")
    print(f"📈 M — Crescendo:   {len(m)} tópicos")
    print(f"🔥 G — Mainstream:  {len(g)} tópicos")
    print(f"\nDetalhes:")
    for t in classificados:
        emoji = {"P": "🌱", "M": "📈", "G": "🔥"}[t["classificacao"]]
        print(f"  {emoji} [{t['classificacao']}] {t['titulo'][:50]} ({t['plataforma']})")

    return classificados

# Executar!
topicos_classificados = classificar_pmg(todos_topicos)


🧠 Classificando tópicos em P / M / G...

🌱 P — Emergentes:  19 tópicos
📈 M — Crescendo:   0 tópicos
🔥 G — Mainstream:  1 tópicos

Detalhes:
  🌱 [P] deborah secco (google_trends)
  🌱 [P] joão sanches (google_trends)
  🌱 [P] memphis depay (google_trends)
  🌱 [P] voto em transito (google_trends)
  🌱 [P] andy burnham (google_trends)
  🌱 [P] clima (google_trends)
  🌱 [P] what (google_trends)
  🌱 [P] wpp (google_trends)
  🌱 [P] star wars (google_trends)
  🌱 [P] paraná pesquisas (google_trends)
  🌱 [P] Separa e Volta - MC Vine7, MC Tuto, MC Ryan SP, MC (youtube)
  🌱 [P] Faces of Grief - Release Date Trailer (youtube)
  🌱 [P] NUNCA VENHA NESSA HAMBURGUERIA! - HAPPY'S HUMBLE B (youtube)
  🌱 [P] ♪ Spider-Noir (Homem-Aranha) | Whisky & Teias | An (youtube)
  🌱 [P] House of the Dragon: S3 - Ep.06 | NEW TRAILER | HB (youtube)
  🌱 [P] PROCURA-SE MESSI - ESPANHA CAMPEÃ!!! FREE FIRE E A (youtube)
  🔥 [G] BTS (방탄소년단) ‘NORMAL’ Official MV (youtube)
  🌱 [P] Akira | Relançamento em 4K | Trailer Oficial 

Continuação Meta

In [233]:
#Bloco 2
import re
import unicodedata

def gerar_hashtags_candidatas(topicos, max_hashtags=10):
    hashtags = []

    for topico in topicos:
        titulo = topico["titulo"]
        titulo_limpo = unicodedata.normalize("NFKD", titulo)
        titulo_limpo = titulo_limpo.encode("ascii", "ignore").decode("utf-8")
        titulo_limpo = re.sub(r"[^a-zA-Z0-9\s]", "", titulo_limpo)
        palavras = titulo_limpo.split()[:2]
        hashtag = "".join(palavras).lower()

        if hashtag and len(hashtag) > 2:
            hashtags.append(hashtag)

    hashtags_unicas = list(dict.fromkeys(hashtags))

    print(f"🏷️  {len(hashtags_unicas)} hashtags candidatas geradas:")
    for h in hashtags_unicas[:max_hashtags]:
        print(f"   #{h}")

    return hashtags_unicas[:max_hashtags]

hashtags_candidatas = gerar_hashtags_candidatas(topicos_classificados)

🏷️  20 hashtags candidatas geradas:
   #deborahsecco
   #joaosanches
   #memphisdepay
   #votoem
   #andyburnham
   #clima
   #what
   #wpp
   #starwars
   #paranapesquisas


In [234]:
#Bloco 3
import requests

def descobrir_instagram_business_id(access_token):
    url = "https://graph.facebook.com/v22.0/me/accounts"
    params = {"access_token": access_token}
    resposta = requests.get(url, params=params).json()

    paginas = resposta.get("data", [])
    if not paginas:
        return None

    page_id = paginas[0]["id"]

    url_ig = f"https://graph.facebook.com/v22.0/{page_id}"
    params_ig = {"fields": "instagram_business_account", "access_token": access_token}
    resposta_ig = requests.get(url_ig, params=params_ig).json()

    ig_account = resposta_ig.get("instagram_business_account", {})
    return ig_account.get("id")


def coletar_meta_hashtags(access_token="", hashtags=None, max_resultados=10):
    if not access_token:
        print("⏳ Meta: credenciais não configuradas — pulando coleta")
        return []

    if not hashtags:
        print("⏳ Meta: nenhuma hashtag candidata recebida — pulando coleta")
        return []

    ig_user_id = descobrir_instagram_business_id(access_token)
    if not ig_user_id:
        print("  Meta erro: não foi possível encontrar conta Instagram Business vinculada")
        return []

    topicos = []

    for hashtag in hashtags:
        try:
            url_busca = "https://graph.facebook.com/v22.0/ig_hashtag_search"
            params_busca = {"user_id": ig_user_id, "q": hashtag, "access_token": access_token}
            resposta_busca = requests.get(url_busca, params=params_busca).json()
            dados_busca = resposta_busca.get("data", [])

            if not dados_busca:
                continue

            hashtag_id = dados_busca[0]["id"]

            url_media = f"https://graph.facebook.com/v22.0/{hashtag_id}/recent_media"
            params_media = {
                "user_id": ig_user_id,
                "fields": "id,caption,like_count,comments_count,timestamp",
                "access_token": access_token
            }
            resposta_media = requests.get(url_media, params=params_media).json()
            posts = resposta_media.get("data", [])[:max_resultados]

            total_likes = sum(p.get("like_count", 0) for p in posts)
            total_comentarios = sum(p.get("comments_count", 0) for p in posts)

            topicos.append({
                "plataforma": "instagram",
                "titulo": f"#{hashtag}",
                "likes": total_likes,
                "comentarios": total_comentarios,
                "posts_encontrados": len(posts)
            })

        except Exception as e:
            print(f"  Meta erro na hashtag #{hashtag}: {e}")

    print(f"✅ Meta: {len(topicos)} hashtags pesquisadas com sucesso")
    return topicos


# Testar!
resultados_meta = coletar_meta_hashtags(
    access_token=META_ACCESS_TOKEN,
    hashtags=hashtags_candidatas
)

  Meta erro: não foi possível encontrar conta Instagram Business vinculada


Chave Claude

In [235]:
%pip install anthropic

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\gablima1\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
ANTHROPIC_API_KEY = "Cole_seu_codigo_aqui"

print("✅ Chave da Anthropic configurada!" if ANTHROPIC_API_KEY else "❌ Chave pendente")

✅ Chave da Anthropic configurada!


In [237]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def gerar_resumo_topico(topico):
    """
    Usa a Claude API para gerar um resumo de contexto
    sobre por que um tópico está em alta.
    """
    titulo = topico["titulo"]
    plataforma = topico["plataforma"]
    classificacao = topico.get("classificacao", "P")

    classificacao_texto = {
        "P": "emergente (começando a aparecer)",
        "M": "crescendo (ganhando força)",
        "G": "mainstream (já é um grande fenômeno)"
    }[classificacao]

    prompt = f"""Você é um analista de social listening. Gere um resumo curto e direto sobre o seguinte tópico em alta:

Tópico: "{titulo}"
Plataforma de origem: {plataforma}
Estágio de crescimento: {classificacao_texto}

Escreva em português, no máximo 3 frases, explicando:
1. Do que provavelmente se trata esse tópico
2. Por que pode estar em alta agora
3. Uma possível oportunidade ou risco para marcas que queiram se posicionar sobre o assunto

Seja direto e evite floreios. Se não tiver certeza do contexto exato, seja honesto sobre isso."""

    try:
        resposta = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}]
        )
        return resposta.content[0].text
    except Exception as e:
        return f"Erro ao gerar resumo: {e}"


# Testar com os 3 primeiros tópicos!
print("🤖 Gerando resumos com IA...\n")

for topico in topicos_classificados[:3]:
    resumo = gerar_resumo_topico(topico)
    emoji = {"P": "🌱", "M": "📈", "G": "🔥"}[topico["classificacao"]]
    print(f"{emoji} [{topico['classificacao']}] {topico['titulo']}")
    print(f"   📝 {resumo}")
    print()

🤖 Gerando resumos com IA...

🌱 [P] deborah secco
   📝 **Resumo — Deborah Secco (Google Trends | Emergente)**

Deborah Secco é atriz e personalidade pública brasileira consolidada, e picos de busca sobre ela geralmente estão ligados a aparições em novelas, entrevistas, polêmicas pessoais ou conteúdos virais nas redes sociais — sem contexto exato disponível, não é possível afirmar a causa específica deste momento. O estágio emergente sugere que algo recente (declaração, post, matéria) começou a gerar curiosidade, mas ainda sem grande volume confirmado. Para marcas, o risco é se associar a um assunto que pode envolver polêmica pessoal ou familiar; a oportunidade existe apenas para marcas de entretenimento, beleza ou lifestyle que já tenham fit natural com o perfil dela, e mesmo assim vale monitorar o contexto antes de qualquer ação.

🌱 [P] joão sanches
   📝 ## Resumo: "joão sanches"

O contexto exato não é claro, pois o nome pode se referir a diferentes pessoas públicas (político, atleta,

In [238]:
import time

def gerar_resumos_em_lote(topicos, limite=None):
    """
    Gera resumos via IA para todos os tópicos (ou um limite definido).
    """
    topicos_para_processar = topicos[:limite] if limite else topicos

    print(f"🤖 Gerando resumos para {len(topicos_para_processar)} tópicos...\n")

    topicos_com_resumo = []

    for i, topico in enumerate(topicos_para_processar, 1):
        resumo = gerar_resumo_topico(topico)

        topico_atualizado = {**topico, "resumo_ia": resumo}
        topicos_com_resumo.append(topico_atualizado)

        emoji = {"P": "🌱", "M": "📈", "G": "🔥"}[topico["classificacao"]]
        print(f"[{i}/{len(topicos_para_processar)}] {emoji} {topico['titulo']}")

        time.sleep(0.5)  # evita estourar rate limit

    print(f"\n✅ {len(topicos_com_resumo)} resumos gerados com sucesso!")
    return topicos_com_resumo


# Rodar para todos os 20 tópicos!
topicos_com_resumo = gerar_resumos_em_lote(topicos_classificados)

🤖 Gerando resumos para 20 tópicos...

[1/20] 🌱 deborah secco
[2/20] 🌱 joão sanches
[3/20] 🌱 memphis depay
[4/20] 🌱 voto em transito
[5/20] 🌱 andy burnham
[6/20] 🌱 clima
[7/20] 🌱 what
[8/20] 🌱 wpp
[9/20] 🌱 star wars
[10/20] 🌱 paraná pesquisas
[11/20] 🌱 Separa e Volta - MC Vine7, MC Tuto, MC Ryan SP, MC Joãozinho VT e MC Poze do Rodo (DJ Gu)
[12/20] 🌱 Faces of Grief - Release Date Trailer
[13/20] 🌱 NUNCA VENHA NESSA HAMBURGUERIA! - HAPPY'S HUMBLE BURGER CULT COM OS AMIGOS
[14/20] 🌱 ♪ Spider-Noir (Homem-Aranha) | Whisky & Teias | AniRap, @TakaB e @AnnyTHN
[15/20] 🌱 House of the Dragon: S3 - Ep.06 | NEW TRAILER | HBO
[16/20] 🌱 PROCURA-SE MESSI - ESPANHA CAMPEÃ!!! FREE FIRE E ALGO MAIS VEM
[17/20] 🔥 BTS (방탄소년단) ‘NORMAL’ Official MV
[18/20] 🌱 Akira | Relançamento em 4K | Trailer Oficial
[19/20] 🌱 Fazendo perguntas BIZARRAS para o Verity
[20/20] 🌱 Dan - SEQUÊNCIA DO MOTOQUEIRO FANTASMA (Ghost Rider)

✅ 20 resumos gerados com sucesso!


In [239]:
for i, topico in enumerate(topicos_com_resumo, 1):
    emoji = {"P": "🌱", "M": "📈", "G": "🔥"}[topico["classificacao"]]
    print(f"{'='*70}")
    print(f"{i}. {emoji} [{topico['classificacao']}] {topico['titulo']}")
    print(f"   📡 Plataforma: {topico['plataforma']}")
    print(f"   📝 Resumo: {topico['resumo_ia']}")
    print()

1. 🌱 [P] deborah secco
   📡 Plataforma: google_trends
   📝 Resumo: ## Resumo: Deborah Secco

O tópico provavelmente está relacionado a alguma aparição recente da atriz em novela, evento, declaração pública ou repercussão em redes sociais — o contexto exato não está disponível aqui. O estágio emergente sugere que algo aconteceu nas últimas horas ou dias e está começando a ganhar tração nas buscas. **Marcas devem aguardar a confirmação do contexto antes de se posicionar**, já que Deborah frequentemente aparece em discussões que vão de entretenimento e moda até assuntos pessoais polêmicos — entrar sem clareza pode gerar associação indesejada.

2. 🌱 [P] joão sanches
   📡 Plataforma: google_trends
   📝 Resumo: **Resumo — "joão sanches" (Google Trends, emergente)**

O nome "João Sanches" pode se referir a uma figura pública em ascensão — possivelmente um político, atleta, influenciador ou personagem de entretenimento —, mas sem contexto adicional não é possível confirmar a identidade exata. 

RECOMENDAÇÃO DE TOPICO

In [240]:
def gerar_recomendacao_marca(topico):
    """
    Usa a Claude API para gerar uma recomendação de oportunidade
    de marca baseada no tópico e seu contexto.
    """
    titulo = topico["titulo"]
    classificacao = topico.get("classificacao", "P")
    resumo = topico.get("resumo_ia", "")

    classificacao_texto = {
        "P": "emergente — ainda dá tempo de ser pioneiro, mas o risco de não decolar é maior",
        "M": "crescendo — janela boa para entrar com timing relevante",
        "G": "mainstream — já é massa crítica, mas a concorrência por atenção é alta"
    }[classificacao]

    prompt = f"""Você é um estrategista de marketing e social listening. Com base no tópico abaixo, recomende se e como uma marca deveria se posicionar sobre ele.

Tópico: "{titulo}"
Estágio: {classificacao_texto}
Contexto: {resumo}

Responda em português, em até 3 frases, no formato:
1. Recomendação: ENTRAR, OBSERVAR ou EVITAR
2. Justificativa curta
3. Se ENTRAR, sugira um formato de conteúdo rápido (ex: post reativo, vídeo curto, colab com criador)

Seja prático e direto, sem floreios."""

    try:
        resposta = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}]
        )
        return resposta.content[0].text
    except Exception as e:
        return f"Erro ao gerar recomendação: {e}"


# Testar com os 3 primeiros!
print("🎯 Gerando recomendações de marca...\n")

for topico in topicos_com_resumo[:3]:
    recomendacao = gerar_recomendacao_marca(topico)
    emoji = {"P": "🌱", "M": "📈", "G": "🔥"}[topico["classificacao"]]
    print(f"{emoji} [{topico['classificacao']}] {topico['titulo']}")
    print(f"   🎯 {recomendacao}")
    print()

🎯 Gerando recomendações de marca...

🌱 [P] deborah secco
   🎯 **1. Recomendação: OBSERVAR**

**2. Justificativa:** O contexto ainda é indefinido — Deborah Secco transita entre moda/beleza e polêmicas pessoais, e entrar sem saber o gatilho do trending pode gerar associação de marca com tema sensível ou irrelevante para o seu público.

**3. Formato (se o contexto se confirmar positivo):** Post reativo com referência visual ao look ou momento dela, ancorando o produto/serviço da marca de forma leve e direta — só faz sentido para marcas de moda, beleza ou entretenimento com fit natural ao perfil dela.

🌱 [P] joão sanches
   🎯 ## Análise: "João Sanches"

1. **Recomendação: OBSERVAR**

2. O nome não tem identidade confirmada nem contexto claro — associar uma marca a uma figura desconhecida ou potencialmente polêmica antes de 48h de monitoramento é risco desnecessário com baixo upside.

3. N/A — aguarde ao menos um ciclo de notícias para identificar se a menção é positiva, quem é o perfil e s

In [241]:
def gerar_recomendacoes_em_lote(topicos, limite=None):
    """
    Gera recomendações de marca via IA para todos os tópicos.
    """
    topicos_para_processar = topicos[:limite] if limite else topicos

    print(f"🎯 Gerando recomendações para {len(topicos_para_processar)} tópicos...\n")

    topicos_completos = []

    for i, topico in enumerate(topicos_para_processar, 1):
        recomendacao = gerar_recomendacao_marca(topico)

        topico_atualizado = {**topico, "recomendacao_marca": recomendacao}
        topicos_completos.append(topico_atualizado)

        emoji = {"P": "🌱", "M": "📈", "G": "🔥"}[topico["classificacao"]]
        print(f"[{i}/{len(topicos_para_processar)}] {emoji} {topico['titulo']}")

        time.sleep(0.5)

    print(f"\n✅ {len(topicos_completos)} recomendações geradas com sucesso!")
    return topicos_completos


# Rodar para todos os 20 tópicos!
topicos_completos = gerar_recomendacoes_em_lote(topicos_com_resumo)

🎯 Gerando recomendações para 20 tópicos...

[1/20] 🌱 deborah secco
[2/20] 🌱 joão sanches
[3/20] 🌱 memphis depay
[4/20] 🌱 voto em transito
[5/20] 🌱 andy burnham
[6/20] 🌱 clima
[7/20] 🌱 what
[8/20] 🌱 wpp
[9/20] 🌱 star wars
[10/20] 🌱 paraná pesquisas
[11/20] 🌱 Separa e Volta - MC Vine7, MC Tuto, MC Ryan SP, MC Joãozinho VT e MC Poze do Rodo (DJ Gu)
[12/20] 🌱 Faces of Grief - Release Date Trailer
[13/20] 🌱 NUNCA VENHA NESSA HAMBURGUERIA! - HAPPY'S HUMBLE BURGER CULT COM OS AMIGOS
[14/20] 🌱 ♪ Spider-Noir (Homem-Aranha) | Whisky & Teias | AniRap, @TakaB e @AnnyTHN
[15/20] 🌱 House of the Dragon: S3 - Ep.06 | NEW TRAILER | HBO
[16/20] 🌱 PROCURA-SE MESSI - ESPANHA CAMPEÃ!!! FREE FIRE E ALGO MAIS VEM
[17/20] 🔥 BTS (방탄소년단) ‘NORMAL’ Official MV
[18/20] 🌱 Akira | Relançamento em 4K | Trailer Oficial
[19/20] 🌱 Fazendo perguntas BIZARRAS para o Verity
[20/20] 🌱 Dan - SEQUÊNCIA DO MOTOQUEIRO FANTASMA (Ghost Rider)

✅ 20 recomendações geradas com sucesso!


In [242]:
for i, topico in enumerate(topicos_completos, 1):
    emoji = {"P": "🌱", "M": "📈", "G": "🔥"}[topico["classificacao"]]
    print(f"{'='*80}")
    print(f"{i}. {emoji} [{topico['classificacao']}] {topico['titulo']}")
    print(f"   📡 Plataforma: {topico['plataforma']}")
    print(f"\n   📝 RESUMO:")
    print(f"   {topico['resumo_ia']}")
    print(f"\n   🎯 RECOMENDAÇÃO DE MARCA:")
    print(f"   {topico['recomendacao_marca']}")
    print()

1. 🌱 [P] deborah secco
   📡 Plataforma: google_trends

   📝 RESUMO:
   ## Resumo: Deborah Secco

O tópico provavelmente está relacionado a alguma aparição recente da atriz em novela, evento, declaração pública ou repercussão em redes sociais — o contexto exato não está disponível aqui. O estágio emergente sugere que algo aconteceu nas últimas horas ou dias e está começando a ganhar tração nas buscas. **Marcas devem aguardar a confirmação do contexto antes de se posicionar**, já que Deborah frequentemente aparece em discussões que vão de entretenimento e moda até assuntos pessoais polêmicos — entrar sem clareza pode gerar associação indesejada.

   🎯 RECOMENDAÇÃO DE MARCA:
   **1. Recomendação: OBSERVAR**

**2. Justificativa:** O contexto do tópico ainda é indefinido — Deborah Secco transita entre moda, entretenimento e polêmicas pessoais, e entrar sem saber o gatilho exato expõe a marca ao risco de associação negativa por um ganho de visibilidade incerto.

**3. Formato sugerido (caso o

Histórico de Código

In [243]:
%pip install prophet

Defaulting to user installation because normal site-packages is not writeable
  Using cached prophet-1.3.0-py3-none-win_amd64.whl.metadata (3.6 kB)
Using cached prophet-1.3.0-py3-none-win_amd64.whl (12.1 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\gablima1\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python312\\site-packages\\prophet\\stan_model\\cmdstan-2.37.0\\stan\\lib\\stan_math\\lib\\tbb_2020.3\\include\\tbb\\internal\\_deprecated_header_message_guard.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\gablima1\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [244]:
import pandas as pd
from prophet import Prophet
import numpy as np
from datetime import datetime, timedelta

def simular_historico_topico(titulo, valor_atual, dias=14, tendencia="crescente"):
    """
    Simula um histórico de 14 dias para um tópico,
    só para demonstrar a mecânica do Prophet.
    Em produção, isso viria de coletas diárias reais salvas no banco.
    """
    datas = [datetime.now() - timedelta(days=i) for i in range(dias, 0, -1)]

    if tendencia == "crescente":
        # Cresce gradualmente até o valor atual
        valores = np.linspace(valor_atual * 0.1, valor_atual, dias)
    else:
        valores = np.linspace(valor_atual * 0.8, valor_atual, dias)

    # Adiciona um pouco de ruído pra parecer mais real
    ruido = np.random.normal(0, valor_atual * 0.05, dias)
    valores = valores + ruido
    valores = np.maximum(valores, 0)  # não deixa ficar negativo

    df = pd.DataFrame({"ds": datas, "y": valores})
    return df


def prever_crescimento(titulo, valor_atual, dias_futuro=7):
    """
    Roda o Prophet em cima do histórico simulado
    e prevê o crescimento dos próximos dias.
    """
    df_historico = simular_historico_topico(titulo, valor_atual)

    modelo = Prophet(daily_seasonality=False, weekly_seasonality=False, yearly_seasonality=False)
    modelo.fit(df_historico)

    futuro = modelo.make_future_dataframe(periods=dias_futuro)
    previsao = modelo.predict(futuro)

    valor_previsto = previsao.iloc[-1]["yhat"]
    crescimento_percentual = ((valor_previsto - valor_atual) / valor_atual) * 100

    return {
        "valor_atual": valor_atual,
        "valor_previsto_7dias": round(valor_previsto, 0),
        "crescimento_percentual": round(crescimento_percentual, 1)
    }


# Testar com um tópico do YouTube (tem métrica numérica real)
topico_teste = next((t for t in topicos_completos if t["plataforma"] == "youtube"), None)

if topico_teste:
    print(f"🔮 Testando previsão para: {topico_teste['titulo']}\n")
    resultado = prever_crescimento(topico_teste["titulo"], topico_teste["metrica_principal"])
    print(f"Valor atual: {resultado['valor_atual']:,.0f}")
    print(f"Previsão em 7 dias: {resultado['valor_previsto_7dias']:,.0f}")
    print(f"Crescimento esperado: {resultado['crescimento_percentual']}%")
else:
    print("Nenhum tópico do YouTube encontrado na lista")

🔮 Testando previsão para: Separa e Volta - MC Vine7, MC Tuto, MC Ryan SP, MC Joãozinho VT e MC Poze do Rodo (DJ Gu)



10:05:49 - cmdstanpy - INFO - Chain [1] start processing
10:05:50 - cmdstanpy - INFO - Chain [1] done processing


Valor atual: 88,112
Previsão em 7 dias: 130,393
Crescimento esperado: 48.0%


Prever volumes crescendo e Emergentes

In [245]:
def prever_volume_emergentes_e_crescendo(topicos):
    """
    Aplica a previsão de crescimento apenas nos tópicos
    classificados como P (Emergente) ou M (Crescendo).
    G (Mainstream) é ignorado, já que já atingiu o pico de alcance.
    """
    topicos_p_m = [t for t in topicos if t["classificacao"] in ["P", "M"]]

    print(f"🔮 Modelando previsão de volume para {len(topicos_p_m)} tópicos (P e M)...\n")

    topicos_com_previsao = []

    for topico in topicos_p_m:
        # Para tópicos sem métrica numérica (ex: Google Trends), usa um valor base simbólico
        valor_base = topico.get("metrica_principal", 100)
        if valor_base == 0:
            valor_base = 100

        try:
            previsao = prever_crescimento(topico["titulo"], valor_base)

            topico_atualizado = {
                **topico,
                "previsao_7dias": previsao["valor_previsto_7dias"],
                "crescimento_previsto_pct": previsao["crescimento_percentual"]
            }
            topicos_com_previsao.append(topico_atualizado)

            emoji = {"P": "🌱", "M": "📈"}[topico["classificacao"]]
            seta = "📈" if previsao["crescimento_percentual"] > 0 else "📉"
            print(f"{emoji} {topico['titulo'][:40]:40} {seta} {previsao['crescimento_percentual']:+.1f}%")

        except Exception as e:
            print(f"  Erro ao prever {topico['titulo']}: {e}")

    print(f"\n✅ {len(topicos_com_previsao)} previsões geradas com sucesso!")
    return topicos_com_previsao


# Rodar!
topicos_com_previsao = prever_volume_emergentes_e_crescendo(topicos_completos)

🔮 Modelando previsão de volume para 19 tópicos (P e M)...



10:05:50 - cmdstanpy - INFO - Chain [1] start processing
10:05:50 - cmdstanpy - INFO - Chain [1] done processing
10:05:50 - cmdstanpy - INFO - Chain [1] start processing


🌱 deborah secco                            📈 +54.2%


10:05:51 - cmdstanpy - INFO - Chain [1] done processing
10:05:51 - cmdstanpy - INFO - Chain [1] start processing


🌱 joão sanches                             📈 +48.8%


...

🌱 PROCURA-SE MESSI - ESPANHA CAMPEÃ!!! FRE 📈 +45.5%


10:05:55 - cmdstanpy - INFO - Chain [1] done processing
10:05:55 - cmdstanpy - INFO - Chain [1] start processing


🌱 Akira | Relançamento em 4K | Trailer Of 📈 +51.5%


10:05:55 - cmdstanpy - INFO - Chain [1] done processing
10:05:56 - cmdstanpy - INFO - Chain [1] start processing
10:05:56 - cmdstanpy - INFO - Chain [1] done processing


🌱 Fazendo perguntas BIZARRAS para o Verity 📈 +54.4%
🌱 Dan - SEQUÊNCIA DO MOTOQUEIRO FANTASMA ( 📈 +40.5%

✅ 19 previsões geradas com sucesso!


Analise de Sentimento Via IA

In [246]:
def analisar_sentimento(topico):
    """
    Usa a Claude API para classificar o sentimento atual
    em torno de um tópico: positivo, negativo ou neutro.
    """
    titulo = topico["titulo"]
    resumo = topico.get("resumo_ia", "")

    prompt = f"""Analise o sentimento público em torno deste tópico em alta:

Tópico: "{titulo}"
Contexto: {resumo}

Responda APENAS com uma palavra: POSITIVO, NEGATIVO ou NEUTRO.
Se não houver informação suficiente para avaliar sentimento, responda NEUTRO."""

    try:
        resposta = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=10,
            messages=[{"role": "user", "content": prompt}]
        )
        sentimento = resposta.content[0].text.strip().upper()
        if sentimento not in ["POSITIVO", "NEGATIVO", "NEUTRO"]:
            sentimento = "NEUTRO"
        return sentimento
    except Exception as e:
        return "NEUTRO"


def analisar_sentimentos_em_lote(topicos):
    print(f"😊 Analisando sentimento de {len(topicos)} tópicos...\n")

    topicos_com_sentimento = []
    for i, topico in enumerate(topicos, 1):
        sentimento = analisar_sentimento(topico)
        topico_atualizado = {**topico, "sentimento": sentimento}
        topicos_com_sentimento.append(topico_atualizado)

        emoji_sentimento = {"POSITIVO": "😊", "NEGATIVO": "😟", "NEUTRO": "😐"}[sentimento]
        print(f"[{i}/{len(topicos)}] {emoji_sentimento} {sentimento:10} — {topico['titulo'][:40]}")

        time.sleep(0.5)

    print(f"\n✅ {len(topicos_com_sentimento)} sentimentos analisados!")
    return topicos_com_sentimento


# Rodar para os tópicos com previsão de volume!
topicos_com_sentimento = analisar_sentimentos_em_lote(topicos_com_previsao)

😊 Analisando sentimento de 19 tópicos...

[1/19] 😐 NEUTRO     — deborah secco
[2/19] 😐 NEUTRO     — joão sanches
[3/19] 😐 NEUTRO     — memphis depay
[4/19] 😐 NEUTRO     — voto em transito
[5/19] 😐 NEUTRO     — andy burnham
[6/19] 😐 NEUTRO     — clima
[7/19] 😐 NEUTRO     — what
[8/19] 😐 NEUTRO     — wpp
[9/19] 😊 POSITIVO   — star wars
[10/19] 😐 NEUTRO     — paraná pesquisas
[11/19] 😊 POSITIVO   — Separa e Volta - MC Vine7, MC Tuto, MC R
[12/19] 😐 NEUTRO     — Faces of Grief - Release Date Trailer
[13/19] 😊 POSITIVO   — NUNCA VENHA NESSA HAMBURGUERIA! - HAPPY'
[14/19] 😊 POSITIVO   — ♪ Spider-Noir (Homem-Aranha) | Whisky & 
[15/19] 😊 POSITIVO   — House of the Dragon: S3 - Ep.06 | NEW TR
[16/19] 😐 NEUTRO     — PROCURA-SE MESSI - ESPANHA CAMPEÃ!!! FRE
[17/19] 😊 POSITIVO   — Akira | Relançamento em 4K | Trailer Of
[18/19] 😐 NEUTRO     — Fazendo perguntas BIZARRAS para o Verity
[19/19] 😊 POSITIVO   — Dan - SEQUÊNCIA DO MOTOQUEIRO FANTASMA (

✅ 19 sentimentos analisados!


Etapa 2 - Previsão de Evolução do sentimento

In [247]:
def prever_sentimento_futuro(topico):
    """
    Usa a Claude API para prever se o sentimento de um tópico
    tende a melhorar, piorar ou se manter, com base no contexto.
    """
    titulo = topico["titulo"]
    sentimento_atual = topico.get("sentimento", "NEUTRO")
    classificacao = topico.get("classificacao", "P")
    resumo = topico.get("resumo_ia", "")

    prompt = f"""Com base no tópico abaixo, preveja a tendência do sentimento público nos próximos dias.

Tópico: "{titulo}"
Sentimento atual: {sentimento_atual}
Estágio de crescimento: {classificacao}
Contexto: {resumo}

Responda APENAS com uma das opções: MELHORANDO, PIORANDO ou ESTAVEL.
Considere que tópicos emergentes com sentimento positivo tendem a se manter ou melhorar conforme ganham força, enquanto tópicos sem contexto claro tendem a ficar estáveis."""

    try:
        resposta = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=10,
            messages=[{"role": "user", "content": prompt}]
        )
        tendencia = resposta.content[0].text.strip().upper()
        if tendencia not in ["MELHORANDO", "PIORANDO", "ESTAVEL"]:
            tendencia = "ESTAVEL"
        return tendencia
    except Exception as e:
        return "ESTAVEL"


def prever_sentimentos_em_lote(topicos):
    print(f"🔮 Prevendo tendência de sentimento para {len(topicos)} tópicos...\n")

    topicos_finais = []
    for i, topico in enumerate(topicos, 1):
        tendencia = prever_sentimento_futuro(topico)
        topico_atualizado = {**topico, "tendencia_sentimento": tendencia}
        topicos_finais.append(topico_atualizado)

        emoji_tendencia = {"MELHORANDO": "📈", "PIORANDO": "📉", "ESTAVEL": "➡️"}[tendencia]
        print(f"[{i}/{len(topicos)}] {emoji_tendencia} {tendencia:10} — {topico['titulo'][:40]}")

        time.sleep(0.5)

    print(f"\n✅ {len(topicos_finais)} tendências de sentimento previstas!")
    return topicos_finais


# Rodar!
topicos_finais = prever_sentimentos_em_lote(topicos_com_sentimento)

🔮 Prevendo tendência de sentimento para 19 tópicos...

[1/19] ➡️ ESTAVEL    — deborah secco
[2/19] ➡️ ESTAVEL    — joão sanches
[3/19] ➡️ ESTAVEL    — memphis depay
[4/19] ➡️ ESTAVEL    — voto em transito
[5/19] ➡️ ESTAVEL    — andy burnham
[6/19] ➡️ ESTAVEL    — clima
[7/19] ➡️ ESTAVEL    — what
[8/19] ➡️ ESTAVEL    — wpp
[9/19] ➡️ ESTAVEL    — star wars
[10/19] ➡️ ESTAVEL    — paraná pesquisas
[11/19] ➡️ ESTAVEL    — Separa e Volta - MC Vine7, MC Tuto, MC R
[12/19] ➡️ ESTAVEL    — Faces of Grief - Release Date Trailer
[13/19] ➡️ ESTAVEL    — NUNCA VENHA NESSA HAMBURGUERIA! - HAPPY'
[14/19] 📈 MELHORANDO — ♪ Spider-Noir (Homem-Aranha) | Whisky & 
[15/19] ➡️ ESTAVEL    — House of the Dragon: S3 - Ep.06 | NEW TR
[16/19] ➡️ ESTAVEL    — PROCURA-SE MESSI - ESPANHA CAMPEÃ!!! FRE
[17/19] 📈 MELHORANDO — Akira | Relançamento em 4K | Trailer Of
[18/19] 📈 MELHORANDO — Fazendo perguntas BIZARRAS para o Verity
[19/19] ➡️ ESTAVEL    — Dan - SEQUÊNCIA DO MOTOQUEIRO FANTASMA (

✅ 19 tendências de se

Consolidar Topicos

In [248]:
def consolidar_topico_final(topico):
    """
    Organiza um tópico no formato final, limpo, para consumo do dashboard.
    """
    return {
        "titulo": topico["titulo"],
        "plataforma": topico["plataforma"],
        "classificacao": topico["classificacao"],
        "resumo": topico.get("resumo_ia", ""),
        "recomendacao_marca": topico.get("recomendacao_marca", ""),
        "sentimento_atual": topico.get("sentimento", "NEUTRO"),
        "tendencia_sentimento": topico.get("tendencia_sentimento", "ESTAVEL"),
        "previsao_volume_7dias": topico.get("previsao_7dias", "N/A"),
        "crescimento_previsto_pct": topico.get("crescimento_previsto_pct", "N/A"),
        "fonte_dados_previsao": "simulado" if "previsao_7dias" in topico else "N/A",
    }


def consolidar_pipeline_final(topicos):
    print(f"🧹 Consolidando {len(topicos)} tópicos no formato final...\n")

    pipeline_final = [consolidar_topico_final(t) for t in topicos]

    print(f"✅ Pipeline finalizado! {len(pipeline_final)} tópicos prontos para o dashboard.")
    return pipeline_final


# Como nem todos os tópicos passaram pela previsão (só P e M),
# juntamos os finais (P/M com previsão) com os G (Mainstream, sem previsão)
topicos_g = [t for t in topicos_completos if t["classificacao"] == "G"]

# Adiciona sentimento/tendência também aos tópicos G, já que isso roda em todos
topicos_g_com_sentimento = []
for topico in topicos_g:
    sentimento = analisar_sentimento(topico)
    tendencia = prever_sentimento_futuro({**topico, "sentimento": sentimento})
    topicos_g_com_sentimento.append({**topico, "sentimento": sentimento, "tendencia_sentimento": tendencia})

pipeline_completo = topicos_finais + topicos_g_com_sentimento

pipeline_final = consolidar_pipeline_final(pipeline_completo)

# Mostrar um exemplo
print("\n📋 Exemplo de tópico consolidado:")
import json
print(json.dumps(pipeline_final[0], indent=2, ensure_ascii=False))

🧹 Consolidando 20 tópicos no formato final...

✅ Pipeline finalizado! 20 tópicos prontos para o dashboard.

📋 Exemplo de tópico consolidado:
{
  "titulo": "deborah secco",
  "plataforma": "google_trends",
  "classificacao": "P",
  "resumo": "## Resumo: Deborah Secco\n\nO tópico provavelmente está relacionado a alguma aparição recente da atriz em novela, evento, declaração pública ou repercussão em redes sociais — o contexto exato não está disponível aqui. O estágio emergente sugere que algo aconteceu nas últimas horas ou dias e está começando a ganhar tração nas buscas. **Marcas devem aguardar a confirmação do contexto antes de se posicionar**, já que Deborah frequentemente aparece em discussões que vão de entretenimento e moda até assuntos pessoais polêmicos — entrar sem clareza pode gerar associação indesejada.",
  "recomendacao_marca": "**1. Recomendação: OBSERVAR**\n\n**2. Justificativa:** O contexto do tópico ainda é indefinido — Deborah Secco transita entre moda, entretenimento e

ARQUIVO JSON

In [ ]:
import json

with open("pipeline_final.json", "w", encoding="utf-8") as f:
    json.dump(pipeline_final, f, ensure_ascii=False, indent=2)

print("✅ pipeline_final.json salvo com sucesso!")

✅ pipeline_final.json salvo com sucesso!


: 